# Diabetes Readmission RL Project
# Approche : Reinforcement Learning Trees (RLT) – Zhu et al., 2015

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
sns.set(style="whitegrid")

## Business Understanding

In [1]:
print("Business Understanding")
print("BO1 — Valider RLT en haute dimension et sparsité")
print(" Évaluer RLT sur datasets variés pour applications industrielles.")
print("BO2 — Optimiser les modèles d'apprentissage")
print("Améliorer la précision et la stabilité des prédictions en ML .")
print("BO3 — Répliquer et étendre la recherche académique")
print("Confirmer les résultats de Zhu et al. (2015) avec implémentation moderne.")

Business Understanding
BO1 — Valider RLT en haute dimension et sparsité
 Évaluer RLT sur datasets variés pour applications industrielles.
BO2 — Optimiser les modèles d'apprentissage
Améliorer la précision et la stabilité des prédictions en ML .
BO3 — Répliquer et étendre la recherche académique
Confirmer les résultats de Zhu et al. (2015) avec implémentation moderne.


## Data Science Objectives

In [2]:
print("\nData Science Objectives")

print("DSO1 —Prédire la cible sur chaque dataset")
print("Régression/Classification avec RLT vs baselines")

print("DSO2 —Comparer les performances")
print("RMSE/Accuracy moyen sur 50 runs.")

print("DSO3 —Analyser l'impact du Variable Muting ")
print("Test avec/sans muting pour confirmer la robustesse.")


Data Science Objectives
DSO1 —Prédire la cible sur chaque dataset
Régression/Classification avec RLT vs baselines
DSO2 —Comparer les performances
RMSE/Accuracy moyen sur 50 runs.
DSO3 —Analyser l'impact du Variable Muting 
Test avec/sans muting pour confirmer la robustesse.


## Data Understanding

 Data Understanding
Les 10 datasets UCI représentent des domaines variés (économie, médecine, environnement). 

| Dataset | Type | Instances | Features | Cible |
|---------|------|-----------|----------|-------|
| Boston Housing | Régression | 506 | 13 | Prix logement |
| Parkinson | Régression | 5 875 | 20 | Score UPDRS |
| Sonar | Classification | 208 | 60 | Mine/Roche |
| White Wine | Régression | 4 898 | 11 | Qualité |
| Red Wine | Régression | 1 599 | 11 | Qualité |
| Parkinson Oxford | Classification | 195 | 22 | Maladie/Sain |
| Ozone | Classification | 2 536 | 73 | Niveau ozone |
| Concrete | Régression | 1 030 | 9 | Résistance |
| Breast Cancer | Classification | 569 | 30 | Malin/Bénin |
| Auto MPG | Régression | 398 | 8 | Consommation |



## Data Preparation

In [6]:
# =====================================================
# CHARGEMENT DES 10 DATASETS UCI (VERSION STABLE)
# =====================================================

import pandas as pd
import numpy as np
import requests
from io import BytesIO
from sklearn.datasets import fetch_openml

all_dataframes = {}
print("Chargement des 10 datasets UCI\n")

# --------------------------------------------------
# 1. Concrete Strength
# --------------------------------------------------
print("Loading concrete ... ", end="")
url_concrete = "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"
content = requests.get(url_concrete).content
df = pd.read_excel(BytesIO(content))
all_dataframes["concrete"] = df
print(f"Success: {df.shape}")

# --------------------------------------------------
# 2. Boston Housing
# --------------------------------------------------
print("Loading boston ... ", end="")
df = pd.read_csv(
    "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
)
all_dataframes["boston"] = df
print(f"Success: {df.shape}")

# --------------------------------------------------
# 3. Wine Quality Red
# --------------------------------------------------
print("Loading wine_red ... ", end="")
df = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv",
    sep=";"
)
all_dataframes["wine_red"] = df
print(f"Success: {df.shape}")

# --------------------------------------------------
# 4. Wine Quality White
# --------------------------------------------------
print("Loading wine_white ... ", end="")
df = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv",
    sep=";"
)
all_dataframes["wine_white"] = df
print(f"Success: {df.shape}")

# --------------------------------------------------
# 5. Parkinson Oxford
# --------------------------------------------------
print("Loading parkinson_oxford ... ", end="")
df = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data"
)
all_dataframes["parkinson_oxford"] = df
print(f"Success: {df.shape}")

# --------------------------------------------------
# 6. Parkinson UPDRS (Telemonitoring)
# --------------------------------------------------
print("Loading parkinson_ml ... ", end="")
df = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/telemonitoring/parkinsons_updrs.data"
)
all_dataframes["parkinson_ml"] = df
print(f"Success: {df.shape}")

# --------------------------------------------------
# 7. Breast Cancer Wisconsin (Diagnostic)
# --------------------------------------------------
print("Loading breast_cancer ... ", end="")
df = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data",
    header=None
)
all_dataframes["breast_cancer"] = df
print(f"Success: {df.shape}")

# --------------------------------------------------
# 8. Sonar
# --------------------------------------------------
print("Loading sonar ... ", end="")
df = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/undocumented/connectionist-bench/sonar/sonar.all-data",
    header=None
)
all_dataframes["sonar"] = df
print(f"Success: {df.shape}")

# --------------------------------------------------
# 9. Auto MPG
# --------------------------------------------------
print("Loading autompg ... ", end="")
url_auto = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data-original"
df = pd.read_fwf(url_auto, header=None, na_values="?")
df = df.iloc[:, :8]  # garder uniquement les colonnes numériques
df.columns = ['mpg', 'cylinders', 'displacement', 'horsepower',
              'weight', 'acceleration', 'model_year', 'origin']
df = df.dropna().reset_index(drop=True)
all_dataframes["autompg"] = df
print(f"Success: {df.shape}")

# --------------------------------------------------
# 10. Ozone Level Detection
# --------------------------------------------------
print("Loading ozone ... ", end="")
try:
    ozone = fetch_openml(data_id=1487, as_frame=True)
    df = ozone.frame.copy()
    df.columns = [f"T{i+1}" for i in range(72)] + ["Class"]
    df["Class"] = df["Class"].astype(int)
except Exception as e:
    print(f"OpenML échoué ({e}), utilisation du backup ...")
    url_backup = "https://raw.githubusercontent.com/datasets/ozone-level-detection/master/data/onehr.data.gz"
    df = pd.read_csv(url_backup, compression="gzip", header=None, na_values="?")
    df.columns = [f"T{i+1}" for i in range(72)] + ["Class"]

all_dataframes["ozone"] = df
print(f"Success: {df.shape}")

# --------------------------------------------------
# Export vers variables globales
# --------------------------------------------------
for name, df in all_dataframes.items():
    globals()[f"df_{name}"] = df

print("\n🎉 TOUS LES 10 DATASETS SONT CHARGÉS CORRECTEMENT !")
print("Formes finales :")
for name, df in all_dataframes.items():
    print(f"  df_{name:<15} → {df.shape}")


Chargement des 10 datasets UCI

Success: (1030, 9).. 
Loading boston ... Success: (506, 14)
Success: (1599, 12). 
Success: (4898, 12)... 
Success: (195, 24)oxford ... 
Success: (5875, 22)l ... 
Success: (569, 32)cer ... 
Success: (208, 61)
Success: (392, 8).. 
Loading ozone ... Success: (2534, 73)

🎉 TOUS LES 10 DATASETS SONT CHARGÉS CORRECTEMENT !
Formes finales :
  df_concrete        → (1030, 9)
  df_boston          → (506, 14)
  df_wine_red        → (1599, 12)
  df_wine_white      → (4898, 12)
  df_parkinson_oxford → (195, 24)
  df_parkinson_ml    → (5875, 22)
  df_breast_cancer   → (569, 32)
  df_sonar           → (208, 61)
  df_autompg         → (392, 8)
  df_ozone           → (2534, 73)


In [13]:
from sklearn.datasets import fetch_california_housing, load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Pipeline de pré-traitement
def pipeline_preprocess(X, y=None, task="regression"):
    X = pd.DataFrame(X).replace("?", np.nan)
    
    # LabelEncoder pour colonnes catégorielles
    for col in X.select_dtypes(include='object').columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
    
    # Imputation médiane
    X = SimpleImputer(strategy='median').fit_transform(X)
    
    # StandardScaler
    X = StandardScaler().fit_transform(X)
    
    if task == "classification" and y is not None:
        y = LabelEncoder().fit_transform(y)
    
    return X, y

# Liste des datasets
datasets = [
    ("California Housing", "regression", fetch_california_housing(return_X_y=True)),
    ("Breast Cancer", "classification", load_breast_cancer(return_X_y=True)),
    ("Wine", "classification", load_wine(return_X_y=True))
]

# Benchmark
results = []
for name, task, data in datasets:
    X, y = data
    X, y = pipeline_preprocess(X, y, task)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
    
    # Random Forest baseline
    model = RandomForestRegressor(n_estimators=50, random_state=42) if task=="regression" else RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    # Score
    if task == "regression":
        score = np.sqrt(mean_squared_error(y_te, y_pred))  # RMSE
    else:
        score = accuracy_score(y_te, y_pred)
    
    results.append((name, score))
    print(f"{name} → Score: {score:.4f}")

print("Benchmark terminé !")


California Housing → Score: 0.5076
Breast Cancer → Score: 0.9708
Wine → Score: 1.0000
Benchmark terminé !


In [ ]:
# =====================================================
## Modelling
# =====================================================

%load_ext rpy2.ipython

%%R
# Chargement du package RLT une seule fois
library(RLT)
options(warn=-1)  # désactiver les warnings temporaires

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.preprocessing import LabelEncoder
import time

# --------------------------------------------------
# Définition des datasets avec cible et type
# --------------------------------------------------
dataset_config = {
    "concrete":        {"target": "Concrete compressive strength(MPa, megapascals) ", "task": "regression", "df": df_concrete},
    "boston":          {"target": "medv", "task": "regression", "df": df_boston},
    "wine_red":        {"target": "quality", "task": "regression", "df": df_wine_red},
    "wine_white":      {"target": "quality", "task": "regression", "df": df_wine_white},
    "parkinson_ml":    {"target": "total_UPDRS", "task": "regression", "df": df_parkinson_ml},
    "autompg":         {"target": "mpg", "task": "regression", "df": df_autompg},
    
    "parkinson_oxford": {"target": "status", "task": "classification", "df": df_parkinson_oxford},
    "breast_cancer":   {"target": 1, "task": "classification", "df": df_breast_cancer},  # colonne 1 = diagnosis
    "sonar":           {"target": 60, "task": "classification", "df": df_sonar},           # dernière colonne
    "ozone":           {"target": "Class", "task": "classification", "df": df_ozone},
}

# Pré-traitement spécifique pour certains datasets
def prepare_dataset(name, df, target_col):
    df = df.copy()
    
    if name == "breast_cancer":
        df.columns = ["id"] + ["diagnosis"] + [f"f{i}" for i in range(30)]
        df["diagnosis"] = (df["diagnosis"] == "M").astype(int)
        target_col = "diagnosis"
    
    if name == "sonar":
        df[60] = df[60].map({"R": 0, "M": 1})
    
    if name == "parkinson_oxford":
        df = df.drop(columns=["name"], errors="ignore")
    
    if name == "concrete":
        target_col = df.columns[-1]  # dernière colonne
    
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # Encodage des variables catégorielles si présentes
    for col in X.select_dtypes(include="object").columns:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    
    return X, y

# --------------------------------------------------
# Fonctions de fitting RLT via rpy2
# --------------------------------------------------
def fit_rlt(X_train, y_train, X_test, task="regression", muting=False):
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.packages import importr
    
    pandas2ri.activate()
    
    ro.globalenv['X_train'] = pandas2ri.py2rpy(pd.DataFrame(X_train))
    ro.globalenv['y_train'] = ro.FloatVector(y_train) if task == "regression" else ro.FactorVector(ro.StrVector(y_train.astype(str)))
    ro.globalenv['X_test']  = pandas2ri.py2rpy(pd.DataFrame(X_test))
    
    ntrees = 200
    mtry = max(1, X_train.shape[1] // 3)
    
    if task == "regression":
        if muting:
            model = ro.r(f'''
                RLTfit(X_train, y_train, 
                       ntrees={ntrees}, mtry={mtry}, 
                       nmin=10, split.gen="best", 
                       reinforcement=TRUE, 
                       variable.muting=TRUE)
            ''')
        else:
            model = ro.r(f'''
                RLTfit(X_train, y_train, 
                       ntrees={ntrees}, mtry={mtry}, 
                       nmin=10, split.gen="best")
            ''')
        
        pred = ro.r(f'predict(model, X_test)$prediction')
        
    else:  # classification
        if muting:
            model = ro.r(f'''
                RLTfit(X_train, y_train, model="classification",
                       ntrees={ntrees}, mtry={mtry}, 
                       nmin=10, split.gen="best", 
                       reinforcement=TRUE, 
                       variable.muting=TRUE)
            ''')
        else:
            model = ro.r(f'''
                RLTfit(X_train, y_train, model="classification",
                       ntrees={ntrees}, mtry={mtry}, 
                       nmin=10, split.gen="best")
            ''')
        
        pred = ro.r(f'predict(model, X_test)$prediction')
    
    return np.array(pred)

# --------------------------------------------------
# Boucle principale d'évaluation (50 runs)
# --------------------------------------------------
n_runs = 50
results = []

print("Démarrage du benchmark RLT vs Baselines sur 10 datasets UCI\n")
print(f"Nombre de runs par configuration : {n_runs}\n")

for name, config in dataset_config.items():
    print(f"Processing {name.upper():<18} ({config['task']}) ...")
    
    df = config["df"]
    target = config["target"]
    task = config["task"]
    
    X, y = prepare_dataset(name, df, target)
    
    # Stockage des scores
    scores = {
        "dataset": name,
        "RLT (no muting)": [], 
        "RLT (with muting)": [],
        "Random Forest": [],
        "Gradient Boosting": [] if task == "regression" else None
    }
    
    for run in range(n_runs):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=run, stratify=y if task=="classification" else None)
        
        # 1. RLT sans muting
        try:
            pred_rlt = fit_rlt(X_train, y_train, X_test, task=task, muting=False)
            if task == "regression":
                score_rlt = mean_squared_error(y_test, pred_rlt, squared=False)  # RMSE
            else:
                score_rlt = accuracy_score(y_test, pred_rlt.round())
            scores["RLT (no muting)"].append(score_rlt)
        except:
            scores["RLT (no muting)"].append(np.nan)
        
        # 2. RLT avec muting
        try:
            pred_rlt_m = fit_rlt(X_train, y_train, X_test, task=task, muting=True)
            if task == "regression":
                score_rlt_m = mean_squared_error(y_test, pred_rlt_m, squared=False)
            else:
                score_rlt_m = accuracy_score(y_test, pred_rlt_m.round())
            scores["RLT (with muting)"].append(score_rlt_m)
        except:
            scores["RLT (with muting)"].append(np.nan)
        
        # 3. Random Forest (sklearn)
        ModelClass = RandomForestRegressor if task == "regression" else RandomForestClassifier
        model_rf = ModelClass(n_estimators=200, max_features="sqrt", random_state=run)
        model_rf.fit(X_train, y_train)
        pred_rf = model_rf.predict(X_test)
        score_rf = mean_squared_error(y_test, pred_rf, squared=False) if task == "regression" else accuracy_score(y_test, pred_rf)
        scores["Random Forest"].append(score_rf)
    
    # Calcul moyenne ± std
    row = {"Dataset": name.capitalize().replace("_", " ")}
    for method in scores:
        if method != "dataset":
            mean_score = np.nanmean(scores[method])
            std_score = np.nanstd(scores[method])
            if task == "classification":
                row[method] = f"{mean_score:.4f} ± {std_score:.4f}"
            else:
                row[method] = f"{mean_score:.3f} ± {std_score:.3f}"
    
    results.append(row)

# --------------------------------------------------
# Affichage des résultats
# --------------------------------------------------
results_df = pd.DataFrame(results)
results_df = results_df.set_index("Dataset")

print("\n" + "="*80)
print("RÉSULTATS FINAUX (moyenne ± écart-type sur 50 runs)")
print("="*80)
display(results_df)

# Export optionnel
results_df.to_csv("rlt_benchmark_results.csv")
print("\nRésultats exportés vers 'rlt_benchmark_results.csv'")